# YOUR FIRST LAB
### Please read this section. This is valuable to get you prepared, even if it's a long read -- it's important stuff.

### Also, be sure to read [README.md](../README.md)!

## Your first Frontier LLM Project


### If necessary, install VSCODE

### Next Select the Kernel

Click on "Select Kernel" on the Top Right

Choose "Python Environments..."

Then choose the one that looks like `.venv (Python 3.12.x) .venv/bin/python` - it should be marked as "Recommended" and have a big star next to it.

Any problems with this? Head over to the troubleshooting.

### Note: you'll need to set the Kernel with every notebook..

In [1]:
# imports

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

# If you get an error running this cell, then please head over to the troubleshooting notebook!

# Connecting to GEMINI (or Ollama)

The next cell is where we load in the environment variables in your `.env` file and connect to GEMINI.  



In [7]:
import os
from dotenv import load_dotenv

load_dotenv()

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    raise ValueError("GOOGLE_API_KEY not found. Check your .env file.")

print("Google API key loaded successfully.")

Google API key loaded successfully.


In [8]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    raise ValueError("GOOGLE_API_KEY not found in .env")

gemini = OpenAI(
    base_url=GEMINI_BASE_URL,
    api_key=google_api_key
)

response = gemini.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages=[
        {"role": "user", "content": "What is 2 + 2?"}
    ]
)

print(response.choices[0].message.content)

2 + 2 = 4


# Let's make a quick call to a Frontier model to get started, as a preview!

In [14]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful technical assistant. Explain concepts clearly with examples."
    },
    {
        "role": "user",
        "content": "Explain what Python generators are and how yield works."
    }
]

In [15]:
#openai = OpenAI()
response = gemini.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages=messages
)

print(response.choices[0].message.content)


To understand Python generators, it helps to first think about how a standard function works.

### The Standard Approach: `return`
Normally, when a function is called, it executes its code, calculates a result, and sends it back using the `return` statement. Once `return` is hit, the function **destroys its local state** and finishes. If you want to return a list of 1 million numbers, the function must calculate all 1 million, store them in memory, and return the entire list at once.

### The Generator Approach: `yield`
A **generator** is a special type of function that allows you to declare a function that behaves like an iterator. It doesn't calculate all its values at once. Instead, it pauses execution and "yields" a value to the caller, saving its state so it can pick up exactly where it left off later.

#### How `yield` works:
1. When the function hits the `yield` keyword, it **pauses** and returns the value to the caller.
2. The function **remembers its local variables** and the 

## OK onwards with our first project

In [16]:
# Let's try out this utility

ed = fetch_website_contents("https://edwarddonner.com")
print(ed)

Home - Edward Donner

Skip to content
Avatar
Curriculum
Proficiency
C4
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of AI startup
Nebula.io
. I was previously founder and CEO of AI startup untapt,
acquired in 2021
, and a Managing Director at JPMorgan.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with over a million enrollments across 194 countries. The
full curriculum is here
. If you’re visiting from one of my courses – I’m super grat

## Types of prompts

You may know this already - but if not, you will get very familiar with it!

Models like GPT have been trained to receive instructions in a particular way.

They expect to receive:

**A system prompt** that tells them what task they are performing and what tone they should use

**A user prompt** -- the conversation starter that they should reply to

In [17]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [18]:
# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

## Messages

The API from OpenAI expects to receive messages in a particular structure.
Many of the other APIs share this structure:

```python
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]
```
To give you a preview, the next 2 cells make a rather simple call - we won't stretch the mighty GPT (yet!)

In [19]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

response = gemini.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages=messages
)

print(response.choices[0].message.content)

2 + 2 = 4


## And now let's build useful messages for Gemini, using a function

In [20]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [21]:
# Try this out, and then try for a few more websites

messages_for(ed)

[{'role': 'system',
  'content': '\nYou are a snarky assistant that analyzes the contents of a website,\nand provides a short, snarky, humorous summary, ignoring text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n'},
 {'role': 'user',
  'content': '\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, then summarize these too.\n\nHome - Edward Donner\n\nSkip to content\nAvatar\nCurriculum\nProficiency\nC4\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of AI startup\nNebula

## Time to bring it together - the API for Gemini is very simple!

In [22]:
def summarize(url):
    website = fetch_website_contents(url)

    response = gemini.chat.completions.create(
        model="gemini-3.1-flash-lite",
        messages=messages_for(website)
    )

    return response.choices[0].message.content

In [23]:
summarize("https://edwarddonner.com")

'### Meet Edward Donner: Professional LLM Evangelist\n\nEd is a self-proclaimed "amateur" musician and professional LLM enthusiast who managed to trick over a million people into taking his Udemy courses. When he’s not busy being a CTO or convincing his friends to stop listening to his AI lectures, he spends his time creating "digital avatars" of himself—presumably so he doesn\'t have to talk to actual humans anymore.\n\n**Latest "Expert" Updates:**\n* **October 2026:** He’s rebranding the "Forward Deployed Engineer" as some sort of "Agentic AI Architect." Sounds fancy, right?\n* **February 2026:** More resources for transitioning from a "Vibe Coder" (writing code based on hope and dreams) to an "Agentic Engineer."\n* **January 2026:** A guide on using n8n to inflict more AI voice agents upon the unsuspecting public.\n* **September 2025:** Apparently, he’s teaching people how to actually put this AI junk into production. Good luck with that.'

In [24]:
# A function to display this nicely in the output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [25]:
display_summary("https://edwarddonner.com")

### Meet Edward Donner: The Man Who Won't Stop Talking About LLMs

Ed is a self-proclaimed serial founder and CTO who realized his friends were tired of his unsolicited AI lectures, so he cleverly pivoted to monetizing his obsession through best-selling Udemy courses instead. When he isn't busy being a "top-rated" educator or running startups, he’s probably on Hacker News pretending to understand what everyone is talking about or building digital arenas for AI models to play gladiators.

**Recent "I’m Still Obsessed" Updates:**
* **October 2026:** He’s now monologuing about being a "Forward Deployed Engineer & Agentic AI Architect."
* **February 2026:** Apparently, you can transition from a "Vibe Coder" to an "Agentic Engineer."
* **January 2026:** He’s teaching people how to make Voice Agents, because the world definitely doesn't have enough robot callers yet.
* **September 2025:** A guide on MLOps, for those who want to see their AI mistakes deployed to production in real-time.

If you somehow haven't had enough, he’s even created a digital avatar of himself so you can chat with a soulless copy of his personality whenever you like. You're welcome.

# Let's try more websites

Note that this will only work on websites that can be scraped using this simplistic approach.

Websites that are rendered with Javascript, like React apps, won't show up. See the community-contributions folder for a Selenium implementation that gets around this. You'll need to read up on installing Selenium (ask ChatGPT!)

Also Websites protected with CloudFront (and similar) may give 403 errors - many thanks Andy J for pointing this out.

But many websites will work just fine!

In [26]:
display_summary("https://cnn.com")

### CNN: The Internet’s Favorite Anxiety Simulator

Congratulations, you’ve arrived at CNN—the digital equivalent of a frantic news ticker duct-taped to a broken video player. 

**Summary:** 
It’s a massive repository of global panic, ranging from geopolitical instability to tips on how to sleep better while the world burns. If you enjoy clicking on buttons that don't load and being asked for feedback on why the ads are freezing your browser, you've found your digital home.

**News & Announcements:**
*   **The World is Still Breaking:** In a shocking turn of events, there are still wars, political elections, and climate crises happening. 
*   **Ad Feedback:** CNN would very much like to know why their website runs like a potato powered by a hamster on a treadmill. Please take a moment to tell them that yes, the video content is indeed failing to load, just like your faith in humanity while browsing their headlines. 
*   **Everything is Content:** Whether it’s the Epstein Files or a listicle about mindfulness, it’s all here, meticulously categorized so you can transition seamlessly from "Global Conflict" to "How to Arrange Your Throw Pillows" in three clicks or less.

In [28]:
# Step 1: Create your prompts

system_prompt = """
You are a helpful AI assistant.
Explain concepts clearly, provide examples, and use simple language.
"""

user_prompt = """
Explain what Large Language Models (LLMs) are.
Describe how they work and give three real-world examples of their applications.
"""

# Step 2: Make the messages list

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

# Step 3: Call Gemini

response = gemini.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages=messages
)

# Step 4: Print the result

print(response.choices[0].message.content)

### What are Large Language Models (LLMs)?

At their simplest, **Large Language Models (LLMs)** are a type of artificial intelligence designed to understand, generate, and manipulate human language. You can think of them as incredibly advanced versions of the "autocomplete" feature on your phone, but instead of predicting the next word in a text message, they can write essays, solve complex problems, and hold conversational chats.

*   **"Large"** refers to both the massive amount of data they are trained on (books, websites, articles, and code) and the billions of internal "parameters" (the mathematical settings) that allow them to process information.
*   **"Language"** refers to their primary function: processing human speech and text.
*   **"Model"** refers to the complex mathematical structure (a neural network) that has been trained to recognize patterns.

---

### How Do They Work?

You can break down how an LLM works into three main stages:

#### 1. Training (Learning the Patte